In [1]:
# Parameters
flux_method = "summed_map"
dig_mode = "dig_subtracted"


In [2]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_candidates = (_here, *_here.parents)
REPO_ROOT = next((candidate for candidate in _candidates if (candidate / "m33_pipeline").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook

REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f"Notebook working directory set to: {REPO_ROOT}")

Notebook working directory set to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1


In [3]:
try:
    flux_method
except NameError:
    flux_method = 'summed_map'
flux_method = 'summed_map'
try:
    dig_mode
except NameError:
    dig_mode = 'dig_subtracted'
dig_mode = 'dig_subtracted'
print(f"Using flux_method={flux_method}, dig_mode={dig_mode}")


from m33_pipeline.config import get_derived_config
from m33_pipeline.derived import (
    add_clustering_metrics,
    add_electron_density,
    add_logU_KK04,
    add_metallicity_columns,
    add_metallicity_error_columns,
    add_boundary_source_flags,
    add_primary_overlap_flags,
    add_peak_region_properties,
    add_symmetry_class,
    add_thermal_pressure,
    merge_field_flux_catalogs,
    write_clustering_outputs,
    write_combined_catalog,
    write_derived_stage_catalog,
    write_total_flux_catalog,
)
from m33_pipeline.validate import validate_total_catalog


Using flux_method=summed_map, dig_mode=dig_subtracted


# Merge per-field flux catalogs


In [4]:
derived_config = get_derived_config()
all_catalog = merge_field_flux_catalogs(method=flux_method, dig_mode=dig_mode)
all_catalog = add_primary_overlap_flags(
    all_catalog,
    match_radius_px=5.0,
    boundary_overlap=True,
    boundary_min_overlap_pixels=1,
    boundary_valid_bounds=(50, 2000, 50, 2000),
)
all_catalog = add_boundary_source_flags(all_catalog, max_zoi_pc=100)
n_overlap_groups = int(all_catalog['is_duplicate_overlap'].fillna(False).groupby(all_catalog['duplicate_group_id']).any().sum()) if 'duplicate_group_id' in all_catalog.columns else 0
n_duplicate_rows = int(all_catalog['is_duplicate_overlap'].fillna(False).sum()) if 'is_duplicate_overlap' in all_catalog.columns else 0
n_non_primary = int((~all_catalog['primary'].fillna(True)).sum()) if 'primary' in all_catalog.columns else 0
n_wr_flagged = int(all_catalog['has_wr_in_boundary'].fillna(False).sum()) if 'has_wr_in_boundary' in all_catalog.columns else 0
n_snr_flagged = int(all_catalog['has_snr_in_boundary'].fillna(False).sum()) if 'has_snr_in_boundary' in all_catalog.columns else 0
print(f"Overlap duplicate groups: {n_overlap_groups}")
print(f"Rows involved in overlap duplicates: {n_duplicate_rows}")
print(f"Rows flagged as non-primary: {n_non_primary}")
print(f"Regions containing WR stars: {n_wr_flagged}")
print(f"Regions containing SNRs: {n_snr_flagged}")
if n_overlap_groups > 0:
    dup_preview_cols = [c for c in ['field', 'region_id', 'duplicate_group_id', 'primary', 'primary_rank', 'primary_region_id', 'duplicate_score_sum_snr'] if c in all_catalog.columns]
    print(all_catalog.loc[all_catalog['is_duplicate_overlap'], dup_preview_cols].sort_values(['duplicate_group_id', 'primary_rank']).head(20).to_string(index=False))
output_path = write_total_flux_catalog(all_catalog, method=flux_method, dig_mode=dig_mode)
print("Flux method:", flux_method)
print("DIG mode:", dig_mode)
print("Combined catalog shape:", all_catalog.shape)
print("Saved combined catalog to:", output_path)
# validate_total_catalog(all_catalog)


Overlap duplicate groups: 134
Rows involved in overlap duplicates: 662
Rows flagged as non-primary: 528
Regions containing WR stars: 25
Regions containing SNRs: 57
field  region_id duplicate_group_id  primary  primary_rank  primary_region_id  duplicate_score_sum_snr
   NE        711           dup_0001     True             1                711              1267.044558
   F5          1           dup_0001    False             2                711              1089.459509
   NE        706           dup_0001    False             3                711               709.281640
   NE        720           dup_0001    False             4                711               618.207190
   NE        722           dup_0001    False             5                711               406.422359
   F5         14           dup_0001    False             6                711               648.966718
   NE        732           dup_0001    False             7                711               306.722015
   NE       

Flux method: summed_map
DIG mode: dig_subtracted
Combined catalog shape: (6518, 351)
Saved combined catalog to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/total_flux_catalog.csv


# Add ionization parameter


In [5]:
cat = add_logU_KK04(all_catalog.copy(), n_mc=derived_config.logu_n_mc, seed=123)
derived_output_path = write_derived_stage_catalog(cat, "ionization_parameter", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(cat.columns))
print("Saved combined catalog with ionization parameter:", derived_output_path)


Number of columns: 376
Saved combined catalog with ionization parameter: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_ionization_parameter.csv


# Add electron density


In [6]:
df = add_electron_density(cat.copy(), n_mc=derived_config.density_n_mc)
df = add_thermal_pressure(
    df,
    T_e=derived_config.electron_temperature_K,
    particle_factor=derived_config.ionized_gas_particle_factor,
)
df = add_peak_region_properties(
    df,
    distance_mpc=derived_config.m33_distance_mpc,
    te_default=derived_config.electron_temperature_K,
    n_mc=derived_config.density_n_mc,
)
derived_output_path = write_derived_stage_catalog(df, "density", method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with electron densities:", derived_output_path)


Number of columns: 432
Saved combined catalog with electron densities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_density.csv


# Add symmetry classification


In [7]:
df = add_symmetry_class(df.copy())
derived_output_path = write_derived_stage_catalog(df, "other_derived", method=flux_method, dig_mode=dig_mode)
print("Symmetry classification counts:")
print(df["symmetry_class"].value_counts())
print("Number of columns:", len(df.columns))
print("Saved combined catalog with other derived properties:", derived_output_path)


Symmetry classification counts:
symmetry_class
asymmetric    5372
symmetric     1146
Name: count, dtype: int64
Number of columns: 433
Saved combined catalog with other derived properties: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_other_derived.csv


# Add metallicity calibrations


In [8]:
df = add_metallicity_columns(df.copy())
df = add_metallicity_error_columns(df.copy(), n_mc=derived_config.metallicity_n_mc, seed=123)
derived_output_path = write_derived_stage_catalog(df, "metallicity", method=flux_method, dig_mode=dig_mode)
combined_output_path = write_combined_catalog(df, method=flux_method, dig_mode=dig_mode)
print("Number of columns:", len(df.columns))
print("Saved combined catalog with metallicities:", derived_output_path)
print("Saved final combined catalog:", combined_output_path)


Number of columns: 485
Saved combined catalog with metallicities: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_metallicity.csv
Saved final combined catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/total_flux_catalog_combined.csv


# Add deprojected clustering metrics


In [9]:
clustered_df, global_stats, ripley_df, pcf_df = add_clustering_metrics(df.copy())
outputs = write_clustering_outputs(clustered_df, global_stats, ripley_df, pcf_df, method=flux_method, dig_mode=dig_mode)
print("Saved catalog:", outputs["catalog"])
print("Saved global stats:", outputs["global"])
print("Saved Ripley profile:", outputs["ripley"])
print("Saved pair-correlation profile:", outputs["pcf"])


Saved catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/total_flux_catalog_clustering.csv
Saved global stats: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/clustering_global_statistics.csv
Saved Ripley profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/clustering_ripley_profile.csv
Saved pair-correlation profile: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/derived/clustering_pair_correlation_profile.csv


In [10]:
# Add region-averaged velocity and velocity-dispersion columns to the published catalog.
# This is intentionally standalone so it can be run after the catalog products already exist.
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits

try:
    flux_method
except NameError:
    flux_method = "summed_map"
try:
    dig_mode
except NameError:
    dig_mode = "dig_subtracted"

_here = Path.cwd().resolve()
_repo_candidates = (_here, *_here.parents)
REPO_ROOT = next(
    (candidate for candidate in _repo_candidates if (candidate / "Boundary_maps").is_dir() and (candidate / "PAPER_PLOTS").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")

CALIBRATED_MAP_ROOT = REPO_ROOT.parent / "M33-Maps-Calibrated"
BOUNDARY_MAP_DIR = REPO_ROOT / "Boundary_maps" / "Boundary_map_100pc"
CATALOG_TABLE_DIR = REPO_ROOT / "PAPER_PLOTS" / flux_method / dig_mode / "tables"
COMBINED_CATALOG_CSV = REPO_ROOT / "CATALOGS" / "flux_catalogs" / flux_method / dig_mode / "total_flux_catalog_combined.csv"
REGION_PROPERTIES_CSV = CATALOG_TABLE_DIR / "M33_emission_region_properties.csv"
REGION_PROPERTIES_COLUMNS_TEX = CATALOG_TABLE_DIR / "M33_emission_region_properties_columns.tex"

KINEMATIC_COLUMNS = [
    "v_helio_mean_kms",
    "v_helio_mean_err_kms",
    "sigma_mean_kms",
    "sigma_mean_err_kms",
    "n_kinematic_spaxels",
]


def _first_existing_path(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError("None of these files exist:\n" + "\n".join(str(c) for c in candidates))


def _calibrated_map_path(field, suffix):
    field = str(field).strip()
    field_dirs = [CALIBRATED_MAP_ROOT / f"M33-{field}"]
    if field == "F9":
        field_dirs.append(CALIBRATED_MAP_ROOT / "M33-F9-not-aligned")
    return _first_existing_path([
        field_dir / f"M33{field}-{suffix}.fits"
        for field_dir in field_dirs
    ])


def _read_map(path):
    data = fits.getdata(path).astype(float)
    data[~np.isfinite(data)] = np.nan
    return data


def _mean_and_mean_error_by_label(label_map, values, errors=None):
    labels = np.asarray(label_map, dtype=int).ravel()
    values = np.asarray(values, dtype=float).ravel()
    valid = (labels > 0) & np.isfinite(values)

    if valid.sum() == 0:
        return np.array([]), np.array([]), np.array([], dtype=int)

    max_label = int(labels[valid].max())
    counts = np.bincount(labels[valid], minlength=max_label + 1)
    sums = np.bincount(labels[valid], weights=values[valid], minlength=max_label + 1)

    means = np.full(max_label + 1, np.nan, dtype=float)
    np.divide(sums, counts, out=means, where=counts > 0)

    mean_errors = np.full(max_label + 1, np.nan, dtype=float)
    if errors is not None:
        errors = np.asarray(errors, dtype=float).ravel()
        valid_errors = valid & np.isfinite(errors)
        if valid_errors.any():
            err_sumsq = np.bincount(
                labels[valid_errors],
                weights=errors[valid_errors]**2,
                minlength=max_label + 1,
            )
            np.divide(np.sqrt(err_sumsq), counts, out=mean_errors, where=counts > 0)

    return means, mean_errors, counts.astype(int)


def _value_at_label(values, label, fill_value=np.nan):
    label = int(label)
    if label < len(values):
        return values[label]
    return fill_value


def build_region_kinematics(region_table):
    rows = []
    for field in sorted(region_table["Field"].dropna().unique()):
        field_rows = region_table.loc[region_table["Field"] == field, ["Field", "ID"]].copy()
        if field_rows.empty:
            continue

        boundary_path = BOUNDARY_MAP_DIR / f"Boundary_map_{field}.fits"
        boundary_map = fits.getdata(boundary_path)
        velocity_map = _read_map(_calibrated_map_path(field, "heliocentric-velocity"))
        velocity_err_map = _read_map(_calibrated_map_path(field, "velocity-err"))
        sigma_map = _read_map(_calibrated_map_path(field, "sigma"))
        sigma_err_map = _read_map(_calibrated_map_path(field, "sigma-err"))

        map_shapes = {arr.shape for arr in [boundary_map, velocity_map, velocity_err_map, sigma_map, sigma_err_map]}
        if len(map_shapes) != 1:
            raise ValueError(f"Map shape mismatch for {field}: {map_shapes}")

        v_means, v_mean_errs, v_counts = _mean_and_mean_error_by_label(
            boundary_map,
            velocity_map,
            velocity_err_map,
        )
        sigma_means, sigma_mean_errs, sigma_counts = _mean_and_mean_error_by_label(
            boundary_map,
            sigma_map,
            sigma_err_map,
        )

        for region_id in field_rows["ID"].astype(int):
            n_vel = _value_at_label(v_counts, region_id, fill_value=0)
            n_sigma = _value_at_label(sigma_counts, region_id, fill_value=0)
            rows.append({
                "Field": field,
                "ID": region_id,
                "v_helio_mean_kms": _value_at_label(v_means, region_id),
                "v_helio_mean_err_kms": _value_at_label(v_mean_errs, region_id),
                "sigma_mean_kms": _value_at_label(sigma_means, region_id),
                "sigma_mean_err_kms": _value_at_label(sigma_mean_errs, region_id),
                "n_kinematic_spaxels": int(min(n_vel, n_sigma)),
            })

    return pd.DataFrame(rows)

def _latex_escape_label(label):
    return str(label).replace("_", r"\_")


def update_combined_catalog_with_kinematics(path, region_kinematics):
    if not path.exists():
        print(f"Combined catalog not found, skipping: {path}")
        return None

    combined = pd.read_csv(path)
    required = {"field", "region_id"}
    if not required.issubset(combined.columns):
        missing = ", ".join(sorted(required.difference(combined.columns)))
        raise KeyError(f"Cannot merge kinematics into {path}; missing column(s): {missing}")

    combined_kinematics = region_kinematics.rename(columns={"Field": "field", "ID": "region_id"})
    combined = combined.drop(columns=KINEMATIC_COLUMNS, errors="ignore")
    combined = combined.merge(
        combined_kinematics,
        on=["field", "region_id"],
        how="left",
        validate="many_to_one",
    )
    combined.to_csv(path, index=False)
    return path


def update_catalog_column_description_tex(path, region_table):
    if not path.exists():
        print(f"Column-description table not found, skipping: {path}")
        return

    text = path.read_text()
    insert_marker = r"\hline" + "\n" + r"\end{longtable}"
    if insert_marker not in text:
        raise ValueError(f"Could not find insertion point in {path}")

    # Remove older versions of these rows before reinserting them, so the cell is safe to rerun.
    lines = text.splitlines()
    filtered_lines = [
        line for line in lines
        if not any(_latex_escape_label(col) in line for col in KINEMATIC_COLUMNS)
    ]
    text = "\n".join(filtered_lines) + "\n"

    descriptions = {
        "v_helio_mean_kms": (
            r"km s$^{-1}$",
            "Mean heliocentric velocity across finite spaxels inside the emission-region boundary.",
        ),
        "v_helio_mean_err_kms": (
            r"km s$^{-1}$",
            "Propagated one-sigma uncertainty on the mean heliocentric velocity.",
        ),
        "sigma_mean_kms": (
            r"km s$^{-1}$",
            "Mean velocity dispersion from the sigma map across finite spaxels inside the emission-region boundary.",
        ),
        "sigma_mean_err_kms": (
            r"km s$^{-1}$",
            "Propagated one-sigma uncertainty on the mean velocity dispersion.",
        ),
        "n_kinematic_spaxels": (
            "spaxels",
            "Number of finite spaxels used for the region-averaged kinematic measurements.",
        ),
    }

    new_rows = []
    for column in KINEMATIC_COLUMNS:
        column_number = int(region_table.columns.get_loc(column)) + 1
        units, description = descriptions[column]
        new_rows.append(
            f"{column_number} & {units} & {_latex_escape_label(column)} & {description} \\\\"
        )
    new_rows_text = "\n".join(new_rows) + "\n"

    text = text.replace(insert_marker, new_rows_text + insert_marker)
    path.write_text(text)


region_properties = pd.read_csv(REGION_PROPERTIES_CSV)
region_kinematics = build_region_kinematics(region_properties)

# Rerun cleanly by replacing any existing kinematic columns before merging.
region_properties = region_properties.drop(columns=KINEMATIC_COLUMNS, errors="ignore")
region_properties = region_properties.merge(region_kinematics, on=["Field", "ID"], how="left", validate="one_to_one")

region_properties.to_csv(REGION_PROPERTIES_CSV, index=False)
updated_combined_catalog_path = update_combined_catalog_with_kinematics(COMBINED_CATALOG_CSV, region_kinematics)
update_catalog_column_description_tex(REGION_PROPERTIES_COLUMNS_TEX, region_properties)

print(f"Updated published catalog table: {REGION_PROPERTIES_CSV}")
if updated_combined_catalog_path is not None:
    print(f"Updated notebook input catalog: {updated_combined_catalog_path}")
print(f"Updated column-description table: {REGION_PROPERTIES_COLUMNS_TEX}")
print(region_properties[KINEMATIC_COLUMNS].describe().to_string())





Updated published catalog table: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/PAPER_PLOTS/summed_map/dig_subtracted/tables/M33_emission_region_properties.csv
Updated notebook input catalog: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/CATALOGS/flux_catalogs/summed_map/dig_subtracted/total_flux_catalog_combined.csv
Updated column-description table: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1/PAPER_PLOTS/summed_map/dig_subtracted/tables/M33_emission_region_properties_columns.tex
       v_helio_mean_kms  v_helio_mean_err_kms  sigma_mean_kms  sigma_mean_err_kms  n_kinematic_spaxels
count       5988.000000           5988.000000     5978.000000         5978.000000          5988.000000
mean        -180.346314              0.475414       17.662753            1.757544           311.163828
std           53.184877              0.418850        6.625505           11.110071           269.245597
min         -312.579446              0.006990        1.339090            0.014928             0.000

## Published Catalog And Catalog Numbers

In [11]:
# Moved from notebook 8: write published emission-region catalog and catalog_numbers.tex.
# This cell prepares notebook-8 compatibility globals for the published-catalog writer below.
from pathlib import Path
import pandas as pd
import numpy as np

catalog_method = flux_method
primary_only = True
cat = clustered_df.copy() if 'clustered_df' in globals() else df.copy()


def _positive_halpha_snr_catalog(df, label='catalog', quiet=False):
    snr_col = next((col for col in [
        'SNR_Halpha_sum',
        'SNR_Halpha_sum_nodig',
        'Halpha_SNR',
        'Halpha_SNR_raw',
        'Halpha_SNR_obs',
    ] if col in df.columns), None)
    if snr_col is None:
        raise KeyError('Cannot apply Halpha S/N > 0 catalog cut: no Halpha S/N column found.')
    snr_values = pd.to_numeric(df[snr_col], errors='coerce')
    out = df.loc[snr_values > 0].copy().reset_index(drop=True)
    if not quiet:
        print(f'Halpha S/N > 0 filtering for {label} using {snr_col}: kept {len(out)} / {len(df)} rows.')
    return out
cat = _positive_halpha_snr_catalog(cat, label='final published catalog')
plot_root = Path('PAPER_PLOTS') / flux_method / dig_mode
plot_root.mkdir(parents=True, exist_ok=True)

def paper_plot_path(filename, subdir=None):
    outdir = plot_root if subdir is None else plot_root / subdir
    outdir.mkdir(parents=True, exist_ok=True)
    return outdir / filename

def load_available_flux_catalog(method, dig_mode, primary_only=True):
    base_dir = Path('CATALOGS') / 'flux_catalogs' / method / dig_mode
    preferred_paths = [base_dir / 'total_flux_catalog_combined.csv', base_dir / 'total_flux_catalog.csv']
    for path in preferred_paths:
        if path.exists():
            out = pd.read_csv(path)
            break
    else:
        field_paths = sorted(base_dir.glob('flux_catalog_*.csv'))
        if not field_paths:
            raise FileNotFoundError(f'No available flux catalogs found in {base_dir}')
        out = pd.concat((pd.read_csv(path) for path in field_paths), ignore_index=True, sort=False)
        path = f'{len(field_paths)} field catalogs from {base_dir}'
    if primary_only and 'primary' in out.columns:
        out = out.loc[out['primary'].fillna(True)].copy().reset_index(drop=True)
    return out, str(path)

def _clean_calibration_label(label):
    import re
    text = str(label)
    match = re.search(r'\(([^()]+?\d{4})\)\s*$', text)
    return match.group(1) if match else text

METALLICITY_GRADIENT_TABLE_ROWS = [
    ('N2', 'Brazzini+2024', r'\citet{Brazzini_2024}'),
    ('N2', 'Brown+2016', r'\citet{Brown_2016}'),
    ('N2', 'Curti+2017', r'\citet{Curti_2017}'),
    ('N2', 'Maiolino+2008', r'\citet{Maiolino_2008}'),
    ('N2', 'M2013', r'\citet{Marino_2013}'),
    ('O3N2', 'Brazzini+2024', r'\citet{Brazzini_2024}'),
    ('O3N2', 'Curti+2017', r'\citet{Curti_2017}'),
    ('O3N2', 'M2013', r'\citet{Marino_2013}'),
    ('R23', 'Curti+2017', r'\citet{Curti_2017}'),
    ('R3', 'Brazzini+2024', r'\citet{Brazzini_2024}'),
]


Halpha S/N > 0 filtering for final published catalog using SNR_Halpha_sum: kept 6516 / 6518 rows.


In [12]:
# Create final CSV catalog with emission-region properties and an AAS-style column description table.
# The CSV expands every available emission line; the TeX description table uses <line>
# placeholder rows for those repeated line-flux columns and lists the actual lines in the caption.

import re


def _first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _series_from(df, col, index=None, default=np.nan):
    if index is None:
        index = df.index
    if col is None or col not in df.columns:
        return pd.Series(default, index=index)
    return df[col]


def _load_no_dig_catalog_for_final_table(reference_cat):
    if 'cat_nodig' in globals():
        out = cat_nodig.copy()
    else:
        out, _ = load_available_flux_catalog(catalog_method, 'no_dig', primary_only=primary_only)
    if primary_only and 'primary' in out.columns:
        out = out.loc[out['primary'].fillna(True)].copy()
    if 'region_id' in out.columns:
        out = out.drop_duplicates('region_id').set_index('region_id')
        return out.reindex(reference_cat['region_id']).reset_index(drop=True)
    return out.reindex(reference_cat.index).reset_index(drop=True)


def _ordered_available_lines(df):
    lines = []
    for col in df.columns:
        match = re.fullmatch(r'F_(.+)_sum(?:_nodig|_digsub)?', col)
        if not match:
            continue
        line = match.group(1)
        if line.endswith('_e'):
            continue
        if line not in lines:
            lines.append(line)
    preferred = ['Halpha', 'Hbeta', '[OIII]5007', '[NII]6583', '[SII]6716', '[SII]6731', '[OII]3727']
    ordered = [line for line in preferred if line in lines]
    ordered.extend([line for line in lines if line not in ordered])
    return ordered


def _clean_column_label(text):
    return str(text).replace('[', '').replace(']', '').replace('_', '')


def _latex_escape_text(text, preserve_citations=True):
    text = str(text)
    protected = []

    if preserve_citations:
        def _protect(match):
            protected.append(match.group(0))
            return f'@@CITE{len(protected) - 1}@@'

        text = re.sub(r'\\citet\{[^}]+\}', _protect, text)

    replacements = {
        '&': r'\&',
        '%': r'\%',
        '#': r'\#',
        '_': r'\_',
        '<': r'$<$',
        '>': r'$>$',
    }
    text = ''.join(replacements.get(char, char) for char in text)

    for idx, citation in enumerate(protected):
        text = text.replace(f'@@CITE{idx}@@', citation)
    return text


def _reference_to_citet(reference):
    reference = str(reference)
    citation_map = {
        'Brazzini': r'\citet{Brazzini_2024}',
        'Brown': r'\citet{Brown_2016}',
        'Curti': r'\citet{Curti_2017}',
        'Dopita': r'\citet{Dopita_2016}',
        'Kobulnicky': r'\citet{Kobulnicky_2004}',
        'Maiolino': r'\citet{Maiolino_2008}',
        'Marino': r'\citet{Marino_2013}',
        'Pettini': r'\citet{Pettini_2004}',
        'Pilyugin': r'\citet{Pilyugin_2016}',
    }
    for key, citation in citation_map.items():
        if key in reference:
            return citation
    return reference


def _add_column(out, descriptions, label, values, units, description):
    out[label] = values
    descriptions.append({
        'label': label,
        'units': units,
        'description': description,
        'columns': [label],
    })


def _add_description_row(descriptions, label, units, description, columns):
    descriptions.append({
        'label': label,
        'units': units,
        'description': description,
        'columns': list(columns),
    })


def _format_column_numbers(columns, table_columns):
    positions = sorted(table_columns.get_loc(col) + 1 for col in columns if col in table_columns)
    if not positions:
        return r'\nodata'
    ranges = []
    start = prev = positions[0]
    for pos in positions[1:]:
        if pos == prev + 1:
            prev = pos
            continue
        ranges.append(f'{start}' if start == prev else f'{start}-{prev}')
        start = prev = pos
    ranges.append(f'{start}' if start == prev else f'{start}-{prev}')
    return ', '.join(ranges)


def _reference_to_column_token(reference):
    return (
        str(reference)
        .replace('+', '')
        .replace(' ', '')
        .replace('.', '')
        .replace('et al', '')
    )


def _selected_metallicity_columns_for_final_table(df):
    rows = []
    requested = globals().get('METALLICITY_GRADIENT_TABLE_ROWS', [])
    for spec in requested:
        indicator, reference = spec[:2]
        citation = spec[2] if len(spec) > 2 else _reference_to_citet(reference)
        reference_token = _reference_to_column_token(reference)
        candidates = [
            f'Z_{indicator}_{reference_token}',
            f'Z_{indicator}_{reference}',
        ]
        match = next((col for col in candidates if col in df.columns), None)
        if match is None:
            match = next(
                (col for col in df.columns
                 if col.startswith(f'Z_{indicator}_')
                 and not col.endswith('_e')
                 and reference_token.lower() in col.lower()),
                None,
            )
        if match is not None:
            rows.append((match, f'{indicator} {citation}'))
    # Keep the user-requested table order while avoiding duplicate calibrations.
    seen = set()
    unique = []
    for col, desc in rows:
        if col in seen:
            continue
        seen.add(col)
        unique.append((col, desc))
    return unique


final_region_cat = cat.copy()
if primary_only and 'primary' in final_region_cat.columns:
    final_region_cat = final_region_cat.loc[final_region_cat['primary'].fillna(True)].copy().reset_index(drop=True)
no_dig_for_table = _load_no_dig_catalog_for_final_table(final_region_cat)

final_table = pd.DataFrame(index=final_region_cat.index)
description_rows = []

_add_column(final_table, description_rows, 'ID', _series_from(final_region_cat, 'region_id'), r'\nodata', 'Emission-region identifier.')
_add_column(final_table, description_rows, 'Field', _series_from(final_region_cat, 'field'), r'\nodata', 'M33 SITELLE field containing the region.')
_add_column(final_table, description_rows, 'RA_deg', _series_from(final_region_cat, 'RA_deg'), 'deg', 'Right ascension of the region center in decimal degrees (J2000).')
_add_column(final_table, description_rows, 'Dec_deg', _series_from(final_region_cat, 'Dec_deg'), 'deg', 'Declination of the region center in decimal degrees (J2000).')
_add_column(final_table, description_rows, 'N_spaxels', _series_from(final_region_cat, 'npix_region'), 'spaxels', 'Number of spaxels assigned to the emission-region boundary.')
_add_column(final_table, description_rows, 'R_c_pc', _series_from(final_region_cat, 'radius_areaeq_pc'), 'pc', 'Circularized equal-area radius from the domain/boundary code.')
_add_column(final_table, description_rows, 'R_16_pc', _series_from(final_region_cat, 'radius_p16_pc'), 'pc', '16th-percentile boundary radius from the domain/boundary code.')
_add_column(final_table, description_rows, 'R_50_pc', _series_from(final_region_cat, 'radius_p50_pc'), 'pc', 'Median boundary radius from the domain/boundary code.')
_add_column(final_table, description_rows, 'R_84_pc', _series_from(final_region_cat, 'radius_p84_pc'), 'pc', '84th-percentile boundary radius from the domain/boundary code.')

available_lines = _ordered_available_lines(final_region_cat)
line_column_labels = []
line_column_descriptions = [
    r'raw integrated flux before extinction correction and DIG/background subtraction',
    r'extinction-corrected integrated flux before DIG/background subtraction',
    r'extinction-corrected integrated flux after DIG/background subtraction',
    r'one-sigma uncertainty on the observed/raw integrated flux before DIG/background subtraction',
    r'signal-to-noise ratio of the observed/raw integrated flux before DIG/background subtraction',
]

for line in available_lines:
    prefix = line
    raw_col = _first_existing_column(no_dig_for_table, [f'F_{line}_sum_nodig', f'F_{line}_sum'])
    dered_col = _first_existing_column(no_dig_for_table, [f'F_{line}_sum_dered_nodig', f'F_{line}_sum_nodig_dered', f'F_{line}_sum_dered'])
    bgsub_col = _first_existing_column(final_region_cat, [f'F_{line}_sum_dered_digsub', f'F_{line}_sum_digsub_dered', f'F_{line}_sum_dered', f'F_{line}_sum_digsub', f'F_{line}_sum'])
    err_col = _first_existing_column(no_dig_for_table, [f'F_{line}_e_sum_nodig', f'F_{line}_e_sum'])
    snr_col = _first_existing_column(no_dig_for_table, [f'SNR_{line}_sum_nodig', f'SNR_{line}_sum'])
    line_labels = [
        f'{prefix}_flux_raw',
        f'{prefix}_flux_dered',
        f'{prefix}_flux_bgsub',
        f'{prefix}_flux_err',
        f'{prefix}_SNR',
    ]
    final_table[line_labels[0]] = _series_from(no_dig_for_table, raw_col, index=final_region_cat.index)
    final_table[line_labels[1]] = _series_from(no_dig_for_table, dered_col, index=final_region_cat.index)
    final_table[line_labels[2]] = _series_from(final_region_cat, bgsub_col, index=final_region_cat.index)
    final_table[line_labels[3]] = _series_from(no_dig_for_table, err_col, index=final_region_cat.index)
    final_table[line_labels[4]] = _series_from(no_dig_for_table, snr_col, index=final_region_cat.index)
    line_column_labels.extend(line_labels)

_add_description_row(
    description_rows,
    '<line>_flux_*; <line>_SNR',
    r'\nodata',
    (
        'Repeated emission-line flux and SNR columns. For each available line, the CSV includes ' +
        '; '.join(line_column_descriptions) +
        '. Flux columns have units erg s$^{-1}$ cm$^{-2}$, while SNR columns are dimensionless.'
    ),
    line_column_labels,
)

_add_column(final_table, description_rows, 'A_V_mag', _series_from(final_region_cat, 'sum_A_V'), 'mag', r'Visual extinction inferred from the Balmer decrement.')
_add_column(final_table, description_rows, 'E_BV_mag', _series_from(final_region_cat, 'sum_E_BV'), 'mag', r'Color excess inferred from the Balmer decrement.')
_add_column(final_table, description_rows, 'BPT_class', _series_from(final_region_cat, 'BPT_class_sum_dered'), r'\nodata', 'BPT classification from extinction-corrected integrated line ratios.')
_add_column(final_table, description_rows, 'L_Halpha_raw', _series_from(no_dig_for_table, 'L_Ha_sum', index=final_region_cat.index), r'erg s$^{-1}$', 'Observed Halpha luminosity before extinction correction and DIG/background subtraction.')
_add_column(final_table, description_rows, 'L_Halpha_dered', _series_from(no_dig_for_table, 'L_Ha_sum_dered', index=final_region_cat.index), r'erg s$^{-1}$', 'Extinction-corrected Halpha luminosity before DIG/background subtraction.')
_add_column(final_table, description_rows, 'L_Halpha_bgsub', _series_from(final_region_cat, 'L_Ha_sum_dered'), r'erg s$^{-1}$', 'Extinction-corrected Halpha luminosity after DIG/background subtraction.')
_add_column(final_table, description_rows, 'has_WR', _series_from(final_region_cat, 'has_wr_in_boundary'), r'\nodata', 'Boolean flag for whether the region boundary contains a cataloged WR star.')
_add_column(final_table, description_rows, 'has_SNR', _series_from(final_region_cat, 'has_snr_in_boundary'), r'\nodata', 'Boolean flag for whether the region boundary contains a cataloged supernova remnant.')
_add_column(final_table, description_rows, 'has_PN', _series_from(final_region_cat, 'has_pn_in_boundary'), r'\nodata', 'Boolean flag for whether the region boundary contains a cataloged planetary nebula.')
_add_column(final_table, description_rows, 'primary', _series_from(final_region_cat, 'primary'), r'\nodata', 'Boolean flag indicating the primary retained region where field overlaps create duplicate detections.')
_add_column(final_table, description_rows, 'v_helio_mean_kms', _series_from(final_region_cat, 'v_helio_mean_kms'), r'km s$^{-1}$', 'Mean heliocentric velocity across finite spaxels inside the emission-region boundary.')
_add_column(final_table, description_rows, 'v_helio_mean_err_kms', _series_from(final_region_cat, 'v_helio_mean_err_kms'), r'km s$^{-1}$', 'Propagated one-sigma uncertainty on the mean heliocentric velocity.')
_add_column(final_table, description_rows, 'sigma_mean_kms', _series_from(final_region_cat, 'sigma_mean_kms'), r'km s$^{-1}$', 'Mean velocity dispersion from the sigma map across finite spaxels inside the emission-region boundary.')
_add_column(final_table, description_rows, 'sigma_mean_err_kms', _series_from(final_region_cat, 'sigma_mean_err_kms'), r'km s$^{-1}$', 'Propagated one-sigma uncertainty on the mean velocity dispersion.')
_add_column(final_table, description_rows, 'n_kinematic_spaxels', _series_from(final_region_cat, 'n_kinematic_spaxels'), 'spaxels', 'Number of finite spaxels used for the region-averaged kinematic measurements.')
_add_column(final_table, description_rows, 'ne', _series_from(final_region_cat, 'ne_SII_cm3'), r'cm$^{-3}$', 'Electron density from the [S II] 6716/6731 ratio.')
_add_column(final_table, description_rows, 'ne_err_minus', _series_from(final_region_cat, 'ne_SII_cm3_mc_minus'), r'cm$^{-3}$', 'Lower one-sigma Monte Carlo uncertainty on the electron density.')
_add_column(final_table, description_rows, 'ne_err_plus', _series_from(final_region_cat, 'ne_SII_cm3_mc_plus'), r'cm$^{-3}$', 'Upper one-sigma Monte Carlo uncertainty on the electron density.')
_add_column(final_table, description_rows, 'Pthermal', _series_from(final_region_cat, 'P_thermal_SII_over_k_K_cm3'), r'K cm$^{-3}$', r'Thermal pressure divided by Boltzmann constant from [S II] electron density.')
_add_column(final_table, description_rows, 'Pthermal_err_minus', _series_from(final_region_cat, 'P_thermal_SII_over_k_K_cm3_mc_minus'), r'K cm$^{-3}$', 'Lower one-sigma Monte Carlo uncertainty on thermal pressure divided by Boltzmann constant.')
_add_column(final_table, description_rows, 'Pthermal_err_plus', _series_from(final_region_cat, 'P_thermal_SII_over_k_K_cm3_mc_plus'), r'K cm$^{-3}$', 'Upper one-sigma Monte Carlo uncertainty on thermal pressure divided by Boltzmann constant.')
_add_column(final_table, description_rows, 'logU_KK04', _series_from(final_region_cat, 'logU_KK04'), 'dex', r'Ionization parameter from the calibration of \citet{Kobulnicky_2004}.')
_add_column(final_table, description_rows, 'logU_KK04_err', _series_from(final_region_cat, 'logU_KK04_e'), 'dex', 'One-sigma uncertainty on the ionization parameter.')

metallicity_columns = _selected_metallicity_columns_for_final_table(final_region_cat)
metallicity_value_labels = []
metallicity_error_labels = []
metallicity_descriptions = []
metallicity_error_series = {}
for col, desc in metallicity_columns:
    final_table[col] = _series_from(final_region_cat, col)
    err_col = f'{col}_e'
    metallicity_value_labels.append(col)
    metallicity_error_labels.append(err_col)
    metallicity_error_series[err_col] = _series_from(final_region_cat, err_col)
    metallicity_descriptions.append(f'{col} ({desc})')

for err_col in metallicity_error_labels:
    final_table[err_col] = metallicity_error_series[err_col]

metallicity_description_text = (
    'Oxygen abundance 12+log(O/H). Available calibration/reference columns: ' +
    '; '.join(metallicity_descriptions) +
    '.'
)
_add_description_row(
    description_rows,
    'Z_<calibration>_<ref>',
    'dex',
    metallicity_description_text,
    metallicity_value_labels,
)
_add_description_row(
    description_rows,
    'Z_err_<calibration>_<ref>',
    'dex',
    'One-sigma uncertainties corresponding to the metallicity columns. Available uncertainty columns: ' + '; '.join(metallicity_error_labels) + '.',
    metallicity_error_labels,
)

final_catalog_csv_path = paper_plot_path('M33_emission_region_properties.csv', subdir='tables')
final_description_tex_path = paper_plot_path('M33_emission_region_properties_columns.tex', subdir='tables')
final_table.to_csv(final_catalog_csv_path, index=False)

line_list_caption = ', '.join(available_lines)
catalog_name = _latex_escape_text(final_catalog_csv_path.name, preserve_citations=False)
caption = (
    r'Description of columns in the machine-readable emission-region property catalog. '
    r'The repeated line-flux columns use the placeholder $<$line$>$; '
    f'the CSV contains these columns for {_latex_escape_text(line_list_caption, preserve_citations=False)}.'
)
continued_caption = r'Description of columns in the machine-readable emission-region property catalog (continued).'
lines = [
    r'% Requires \usepackage{longtable}',
    r'\begingroup',
    r'\small',
    r'\begin{longtable}{r p{0.13\textwidth} p{0.23\textwidth} p{0.50\textwidth}}',
    rf'\caption{{{caption}\label{{tab:catalog_columns}}}}\\',
    r'\hline',
    r'Column number & Units & Label & Description \\',
    r'\hline',
    r'\endfirsthead',
    rf'\caption[]{{{continued_caption}}}\\',
    r'\hline',
    r'Column number & Units & Label & Description \\',
    r'\hline',
    r'\endhead',
    r'\hline',
    r'\multicolumn{4}{r}{Continued on next page}\\',
    r'\endfoot',
    r'\hline',
    r'\endlastfoot',
]
for row in description_rows:
    column_numbers = _format_column_numbers(row['columns'], final_table.columns)
    label_tex = _latex_escape_text(row['label'], preserve_citations=False)
    description_tex = _latex_escape_text(row['description'], preserve_citations=True)
    lines.append(f'{column_numbers} & {row["units"]} & {label_tex} & {description_tex} ' + r'\\')
lines.extend([
    r'\hline',
    # rf'\multicolumn{{4}}{{p{{0.93\textwidth}}}}{{\footnotesize The full CSV catalog is written to \texttt{{{catalog_name}}}. Fluxes are integrated over each emission-region boundary.}}\\',
    r'\end{longtable}',
    r'\endgroup',
    '',
])

final_description_tex_path.write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote final emission-region CSV catalog: {final_catalog_csv_path} ({len(final_table)} rows, {len(final_table.columns)} columns)')
print(f'Wrote final emission-region column description table: {final_description_tex_path}')
print(f'Available emission lines included in CSV: {line_list_caption}')




Wrote final emission-region CSV catalog: PAPER_PLOTS/summed_map/dig_subtracted/tables/M33_emission_region_properties.csv (5988 rows, 87 columns)
Wrote final emission-region column description table: PAPER_PLOTS/summed_map/dig_subtracted/tables/M33_emission_region_properties_columns.tex
Available emission lines included in CSV: Halpha, Hbeta, [OIII]5007, [NII]6583, [SII]6716, [SII]6731, [OII]3727


In [13]:
# Generate catalog_numbers.tex from the final combined catalog.
from m33_pipeline.reporting import (
    build_catalog_number_values,
    add_duplicate_region_values,
    write_latex_commands,
)

numbers_cat = clustered_df.copy() if 'clustered_df' in globals() else df.copy()
numbers_cat = _positive_halpha_snr_catalog(numbers_cat, label='catalog_numbers.tex input')
values, formats = build_catalog_number_values(numbers_cat, snr_cut=3)
if 'all_catalog' in globals():
    duplicate_numbers_cat = _positive_halpha_snr_catalog(
        all_catalog,
        label='duplicate-region catalog_numbers.tex input',
        quiet=True,
    )
    add_duplicate_region_values(values, duplicate_numbers_cat)
catalog_numbers_path = paper_plot_path('catalog_numbers.tex')
write_latex_commands(catalog_numbers_path, values, formats)
print(f'Wrote {len(values)} auto-generated catalog commands to {catalog_numbers_path}')


Halpha S/N > 0 filtering for catalog_numbers.tex input using SNR_Halpha_sum: kept 6516 / 6518 rows.


Wrote 211 auto-generated catalog commands to PAPER_PLOTS/summed_map/dig_subtracted/catalog_numbers.tex
